# 02 Logistic Regression (Harry Verdeflor)

Model 1 of 3 in HeatCast NCR. Logistic Regression is the simple, explainable baseline.

**Rules:** choose everything with the 5 time-ordered folds inside the training days. Do not look at the test days here. The group runs `final_test()` once, for the model it picks.

In [ ]:
import sys; sys.path.append("../src")
from common import *
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = load_data()
train, test = split_train_test(df)   # test stays untouched in this notebook
print(len(train), "training rows;", train.Date.min().date(), "to", train.Date.max().date())

## 1. Baseline

Standard scaling of the five weather numbers and one-hot City, then Logistic Regression with default `C`.

In [ ]:
baseline = make_pipeline(make_preprocess(scale=True), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
baseline_scores = cross_validate(baseline, train)
print("Logistic Regression, default settings")
display(summarize(baseline_scores))
print("Copy-today rule (must be beaten)")
display(summarize(persistence_cv(train)))

## 2. Tuning (5 folds only)

Try a few values of `C` (regularisation strength) and class weights. Choose by `macro_f1_core3` and `danger_recall`.

In [ ]:
rows = []
for C in [0.01, 0.1, 1, 10]:
    for cw in [None, "balanced"]:
        model = make_pipeline(make_preprocess(scale=True),
                              LogisticRegression(C=C, class_weight=cw, max_iter=1000, random_state=RANDOM_STATE))
        mean = summarize(cross_validate(model, train))["mean"]   # mean over the 5 folds
        rows.append({"C": C, "class_weight": str(cw), **mean.to_dict()})

table = pd.DataFrame(rows).set_index(["C", "class_weight"])
display(table.round(3))

Record the best setting here and why, once you have run it:

- Best setting: `C=10`, no class weights, plain features (no log1p).
- Why: highest core-3 macro F1 (0.632 ± 0.043) and highest Danger recall among the settings without class weights (0.276 ± 0.101), with the best Danger precision of the grid (0.655). Balanced weights reach about 0.87 Danger recall but drop precision to about 0.31 and core-3 macro F1 to about 0.61, so they flag many more false Danger days. C=1 is within one standard deviation of C=10 (0.628 ± 0.046) and is a reasonable alternative.

## 2b. Option: log(1 + rainfall)

The EDA (E5) says to log-transform `Rainfall_Total` for Logistic Regression: skew falls from 5.45 to 0.73. `make_preprocess()` does not do this yet, so this cell builds a version that does. It is a separate option; section 2 above is unchanged. Compare the two tables and record which one you used.

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

def make_preprocess_log(scale=True):
    """Like make_preprocess(), but Rainfall_Total gets log(1 + x) before scaling."""
    other = [c for c in NUMERIC if c != "Rainfall_Total"]
    return ColumnTransformer([
        ("rain", make_pipeline(FunctionTransformer(np.log1p), StandardScaler() if scale else FunctionTransformer()), ["Rainfall_Total"]),
        ("numbers", StandardScaler() if scale else "passthrough", other),
        ("city", OneHotEncoder(handle_unknown="ignore"), ["City"]),
    ])

rows_log = []
for C in [0.01, 0.1, 1, 10]:
    for cw in [None, "balanced"]:
        model = make_pipeline(make_preprocess_log(scale=True),
                              LogisticRegression(C=C, class_weight=cw, max_iter=1000, random_state=RANDOM_STATE))
        mean = summarize(cross_validate(model, train))["mean"]
        rows_log.append({"C": C, "class_weight": str(cw), **mean.to_dict()})

table_log = pd.DataFrame(rows_log).set_index(["C", "class_weight"])
display(table_log.round(3))

## 3. Confusion matrix for the chosen setting

Set `CHOSEN` to the setting you picked above. The matrix is summed over the 5 check blocks.

In [ ]:
CHOSEN = {"C": 10, "class_weight": None}   # picked from the table in section 2 (plain features, no log1p)

LABELS = ["Caution", "Extreme Caution", "Danger"]   # the three levels with enough days
chosen = make_pipeline(make_preprocess(scale=True),
                       LogisticRegression(C=CHOSEN["C"], class_weight=CHOSEN["class_weight"],
                                          max_iter=1000, random_state=RANDOM_STATE))
total = pd.DataFrame(0, index=LABELS, columns=LABELS)
X, y = train[FEATURES], train[TARGET]
for fit_idx, check_idx in make_folds(train):
    m = clone(chosen).fit(X.iloc[fit_idx], y.iloc[fit_idx])
    cm = confusion_matrix(y.iloc[check_idx], m.predict(X.iloc[check_idx]), labels=LABELS)
    total += cm
display(total)

## 4. Findings (for Section 5.1 of the Group Workspace)

**What I tried.** Logistic Regression with the five weather inputs standardised and City as 16 yes/no columns. I varied `C` over 0.01, 0.1, 1 and 10, and class weights over none and balanced. I ran each setting with the plain rainfall input and with log(1 + rainfall). All scores come from the 5 time-ordered folds inside the training days (2021-01-01 to 2024-12-27). The test days were not used.

**Chosen setting:** `C = 10`, no class weights, plain rainfall input.

**5-fold scores, chosen setting (mean ± SD), next to the copy-today rule:**

| Score | Logistic Regression | Copy-today rule |
|---|---|---|
| Macro F1, all levels | 0.569 ± 0.095 | 0.620 ± 0.102 |
| Macro F1, core 3 | 0.632 ± 0.043 | 0.679 ± 0.051 |
| Accuracy | 0.858 ± 0.038 | 0.846 ± 0.039 |
| Danger recall | 0.276 ± 0.101 | 0.518 ± 0.153 |
| Danger precision | 0.655 ± 0.158 | 0.527 ± 0.153 |

**Confusion matrix** (rows = true level, columns = predicted level, summed over the 5 check blocks): see section 3.

**What I learned.**
- At the chosen setting the model does **not** beat the copy-today rule on the two scores we care about most. Core-3 macro F1 is 0.632 against 0.679, and Danger recall is 0.276 against 0.518. Only accuracy and Danger precision are higher.
- The model misses most Danger days. In the pooled matrix, 1,041 of 1,553 true Danger days are predicted Extreme Caution, so recall is about 33%. That is the costly error for the warning use case.
- It raises few false alarms. Only 202 Extreme Caution days are predicted as Danger, so precision is about 72%, and it rarely confuses Caution with Danger.
- Balanced class weights catch far more Danger days (recall about 0.87, above the copy-today rule's 0.518) but drop precision to about 0.31, and their best core-3 macro F1 is 0.612, below the chosen setting (0.632) and below both tree models (Random Forest 0.653, Gradient Boosting 0.643). Under the two-step rule proposed in Section 6.1 they pass Step 1 but lose at Step 2.
- The log transform on rainfall changes the scores by at most about 0.01, so it is not what decides the result.
- Fold variation is large for Danger (recall SD about 0.10), because Danger is seasonal and differs from year to year.

**Implication:** Logistic Regression is a useful, explainable baseline, but on these folds it does not improve on the copy-today rule for Danger. Under the two-step rule proposed in Section 6.1, the chosen setting fails Step 1 (Danger recall 0.276 against 0.518) and the balanced settings lose at Step 2, so Section 6.3 proposes Random Forest instead. (These are validation-fold results. The test days have not been checked.)

**Not done here, on purpose:** `final_test()` runs once, for the model the group picks.